# HOMEWORK 13

In this homework you are going to build your first classifier for the CIFAR-10 dataset. This dataset contains 10 different classes and you can learn more about it [here](https://www.cs.toronto.edu/~kriz/cifar.html). This homework consists of the following tasks:
* Dataset inspection
* Building the network
* Training
* Evaluation

At the end, as usual, there will be a couple of questions for you to answer :-)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D
from tensorflow.keras import Model
from time import time

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seeds for reproducibility
from numpy.random import seed
from tensorflow.random import set_seed
seed_value = 1234578790
seed(seed_value)
set_seed(seed_value)

### Step 0: Dataset Inspection

Load the dataset and make a quick inspection.

In [ ]:
# Загружаем датасет
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
# Названия классов
classes = {0:'plane', 1:'car', 2:'bird', 3:'cat', 4:'deer',
           5:'dog', 6:'frog', 7:'horse', 8:'ship', 9:'truck'}

num_classes = len(classes)
size = x_train.shape[1]

# Показываем случайные картинки
for ii in range(18):    
    plt.subplot(3,6,ii+1)
    idx = np.random.randint(0, len(x_train))
    plt.imshow(x_train[idx, ...])
    plt.title(classes[int(y_train[idx])])

Compute the class histogram (you can visualize it if you want). Is the dataset balanced?

Hint: You might find [Counter](https://docs.python.org/3/library/collections.html#collections.Counter) tool useful. In any case, it's up to you how you compute the histogram.

In [ ]:
# гистограмма классов
from collections import Counter

counts = Counter(y_train.flatten())
print("Сколько картинок в каждом классе:")
for class_id in sorted(counts.keys()):
    print(f"  {classes[class_id]}: {counts[class_id]}")

plt.bar([classes[i] for i in range(num_classes)], [counts[i] for i in range(num_classes)])
plt.ylabel('count')
plt.title('classes')
plt.show()

# датасет сбалансированный, везде по 5000

### Step 1: Data Preparation

In this step, you'll need to prepare the data for training, i.e., you will have to normalize it and encode the labels as one-hot vectors.

In [ ]:
# Нормализация
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# One-hot кодирование меток
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

### Step 2: Building the Classifier

Build the CNN for CIFAR10 classification. For starters, you can use the same network we used in the lesson for the MNIST problem.

In [ ]:
# Строим модель CNN
inputs = Input(shape=(size, size, 3))

x = Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
x = MaxPooling2D((2,2))(x)
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)

model.summary()

### Step 3: Training

Compile the model and train it.

In [ ]:
epochs = 25
batch_size = 128

# Компилируем модель
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Обучаем
history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_split=0.2)

In [ ]:
# Show training history (this cell is complete, nothing to implement here :-) )
h = history.history
epochs = range(len(h['loss']))

plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
plt.legend(['Train', 'Validation'])
plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                           epochs, h['val_accuracy'], '.-')
plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])

print('Train Acc     ', h['accuracy'][-1])
print('Validation Acc', h['val_accuracy'][-1])    

### Step 4: Evaluation

In this step, you have to calculate the accuracies and visualize some random samples. For the evaluation, you are going to use the test split from the dataset.

In [ ]:
# Получаем предсказания
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model.predict(x_test), axis=1)

In [ ]:
# Точность для каждого класса
for class_id, class_name in classes.items():
    mask = (y_true == class_id)
    acc = np.mean(y_pred[mask] == y_true[mask])
    print(class_name, f'{acc:.4f}')

In [ ]:
# Print the overall stats
ev = model.evaluate(x_test, y_test)
print('Test loss  ', ev[0])
print('Test metric', ev[1])

In [ ]:
# Показываем случайные примеры с предсказаниями
for ii in range(15):
    idx = np.random.randint(0, len(x_test))
    plt.subplot(3,5,ii+1), plt.imshow(x_test[idx, ...])
    plt.title('True: ' + str(classes[y_true[idx]]) + ' | Pred: ' + str(classes[y_pred[idx]]))

### Вопросы

1. Общая точность где-то 70%, точное значение выше в evaluate.

2. Для улучшения можно попробовать:
   - добавить еще сверточных слоев
   - добавить Dropout от переобучения
   - BatchNormalization
   - аугментацию данных

3. Я добавила Dropout и еще один сверточный слой, результат ниже.

In [ ]:
from tensorflow.keras.layers import Dropout

# улучшенная модель с Dropout
inputs2 = Input(shape=(size, size, 3))

x = Conv2D(32, (3,3), activation='relu', padding='same')(inputs2)
x = Conv2D(32, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Dropout(0.25)(x)

x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Dropout(0.25)(x)

x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
outputs2 = Dense(num_classes, activation='softmax')(x)

model2 = Model(inputs2, outputs2)

model2.compile(optimizer='adam',
               loss='categorical_crossentropy',
               metrics=['accuracy'])

history2 = model2.fit(x_train, y_train,
                      batch_size=128,
                      epochs=25,
                      validation_split=0.2)

model2.evaluate(x_test, y_test)
# стало лучше, около 75-78%